# 03 - Silver: RUES (Registro Mercantil)

Trabajo Práctico - Pipeline ETL end-to-end (Bronze -> Silver / DQL -> Gold)

| | Tabla |
|---|---|
| Entrada | `datos_empresas.bronze.de_semiestructurado_registromercantil_api` + `datos_empresas.silver.de_actividadeseconomicas` (para la regla de correspondencia) |
| Salida Silver | `datos_empresas.silver.de_registromercantil` |
| Salida DQL | `datos_empresas.dql.de_registromercantil_dql` |
| Llave (grano) | `codigo_camara` + `matricula` (una matrícula es única dentro de su cámara de comercio) |

**Depende de `02_Silver_CIIU`.**

## Transformaciones

* **Fechas** `fecha_matricula`, `fecha_renovacion`, `fecha_cancelacion`, `fecha_vigencia`: texto `yyyyMMdd` -> `DATE` con `try_to_timestamp` (las fechas imposibles, como un mes 99, quedan `NULL` en vez de romper el proceso con ANSI activado). En `fecha_vigencia`, el centinela `99991231` ("sin vencimiento") se convierte en `NULL`.
* **`fecha_actualizacion`**: texto `yyyy/MM/dd HH:mm:ss.SSSSSSSSS` -> `TIMESTAMP`. Define cuál es la versión más reciente de cada empresa.
* **Identificación**: `numero_identificacion_normalizado` y `nit_normalizado` contienen solo dígitos y no tienen ceros a la izquierda. Son la llave para cruzar con los proveedores de SECOP en Gold.
* **Categorías** (`estado_matricula`, `organizacion_juridica`, `tipo_sociedad`, `categoria_matricula`, `razon_social`): `trim` + mayúsculas para estandarizar.
* **Minimización de datos personales**: no pasan a Silver los nombres ni el documento del representante legal, ni los nombres y apellidos de las personas naturales. No se usan en el análisis y siguen disponibles en Bronze.

## Reglas de calidad (rechazo hacia DQL)

| # | Regla (`_motivo_rechazo`) | Condición | Por qué es pertinente |
|---|---|---|---|
| 1 | `campo_obligatorio_vacio` | `codigo_camara`, `matricula`, `razon_social` o `estado_matricula` vacíos | Son la llave de la empresa y los atributos mínimos para identificarla y saber si está activa. Sin ellos el registro no se puede usar ni enlazar en Gold. |
| 2 | `identificacion_invalida` | `numero_identificacion` vacío o compuesto solo de ceros (`0000000000000`) | La identificación (NIT o cédula) es el único dato que conecta una empresa con sus contratos en SECOP. Un documento en ceros es un valor de relleno de la fuente, no una identificación real. |
| 3 | `fecha_matricula_invalida` | Fecha mal formada, anterior a 1900 (por ejemplo `00010101`) o posterior a hoy | La fecha de matrícula es la base de los indicadores de creación de empresas en Gold. Una fecha imposible distorsiona las series mensuales. Ya se detectaba como valor atípico en las validaciones del TP1. |
| 4 | `fecha_cancelacion_anterior_a_matricula` | `fecha_cancelacion` < `fecha_matricula` | Una empresa no puede cancelarse antes de existir. Esta inconsistencia invalidaría el cálculo de la vida de la empresa (`anios_hasta_cancelacion`). |
| 5 | `ciiu_no_existe_en_catalogo` | El CIIU principal está vacío o no existe en `silver.de_actividadeseconomicas` | **Regla de correspondencia**: Gold clasifica cada empresa por su actividad económica. Si el código no existe en el catálogo, la empresa quedaría sin clasificar. Además, DQL deja ver qué códigos le faltan al catálogo (por ejemplo `9999` y `7112`) para corregirlo. |
| 6 | `duplicado_version_anterior` | Hay una versión más reciente de la misma `codigo_camara` + `matricula` | **Unicidad**: Bronze se carga de forma incremental, así que una empresa actualizada llega otra vez. Se conserva la versión con mayor `fecha_actualizacion`; las anteriores quedan en DQL como historial. |

In [ ]:
%run ./00_Utilidades_Pipeline

---
## Paso 1: Lectura de Bronze, tipado y estandarización

In [ ]:
%python
FORMATO_FECHA_ACTUALIZACION = "yyyy/MM/dd HH:mm:ss.SSSSSSSSS"

df_bronze = preparar_bronze(spark.table(BRONZE_RUES))

df_tipado = df_bronze.select(
    "_registro_bronze", "_hash_bronze", "_id_registro_bronze",
    F.col("_ingested_at").alias("_bronze_ingested_at"),
    texto_limpio("codigo_camara").alias("codigo_camara"),
    texto_limpio("camara_comercio", mayusculas=True).alias("camara_comercio"),
    texto_limpio("matricula").alias("matricula"),
    texto_limpio("razon_social", mayusculas=True).alias("razon_social"),
    texto_limpio("sigla", mayusculas=True).alias("sigla"),
    texto_limpio("clase_identificacion", mayusculas=True).alias("clase_identificacion"),
    texto_limpio("numero_identificacion").alias("numero_identificacion"),
    solo_digitos("numero_identificacion").alias("numero_identificacion_normalizado"),
    texto_limpio("nit").alias("nit"),
    texto_limpio("digito_verificacion").alias("digito_verificacion"),
    solo_digitos("nit").alias("nit_normalizado"),
    texto_limpio("cod_ciiu_act_econ_pri").alias("codigo_ciiu_principal"),
    texto_limpio("cod_ciiu_act_econ_sec").alias("codigo_ciiu_secundario"),
    fecha_yyyymmdd("fecha_matricula").alias("fecha_matricula"),
    fecha_yyyymmdd("fecha_renovacion").alias("fecha_renovacion"),
    F.expr("try_cast(ultimo_ano_renovado AS INT)").alias("ultimo_ano_renovado"),
    F.when(F.trim("fecha_vigencia") == "99991231", F.lit(None).cast("date"))
     .otherwise(fecha_yyyymmdd("fecha_vigencia")).alias("fecha_vigencia"),
    fecha_yyyymmdd("fecha_cancelacion").alias("fecha_cancelacion"),
    texto_limpio("tipo_sociedad", mayusculas=True).alias("tipo_sociedad"),
    texto_limpio("organizacion_juridica", mayusculas=True).alias("organizacion_juridica"),
    texto_limpio("categoria_matricula", mayusculas=True).alias("categoria_matricula"),
    texto_limpio("estado_matricula", mayusculas=True).alias("estado_matricula"),
    texto_limpio("inscripcion_proponente").alias("inscripcion_proponente"),
    F.expr(f"try_to_timestamp(fecha_actualizacion, '{FORMATO_FECHA_ACTUALIZACION}')").alias("fecha_actualizacion"),
)

# Catálogo CIIU ya validado en Silver (regla de correspondencia)
df_catalogo = spark.table(SILVER_CIIU).select(F.col("codigo_ciiu").alias("_ciiu_en_catalogo"))
df_tipado = df_tipado.join(
    F.broadcast(df_catalogo), df_tipado["codigo_ciiu_principal"] == df_catalogo["_ciiu_en_catalogo"], "left"
)

df_versionado = marcar_version(
    df_tipado, ["codigo_camara", "matricula"],
    [F.col("fecha_actualizacion").desc_nulls_last(), F.col("_bronze_ingested_at").desc()],
)

---
## Paso 1.5: Diagnóstico previo — evidencia detrás de cada regla

Antes de rechazar, se mide sobre Bronze cuántas filas incumpliría cada regla. Una fila puede incumplir varias reglas, así que las columnas no se suman. Resultado con el Bronze del 21-oct-2026 (33.324 filas): `identificacion_invalida` = 1.903, `ciiu_no_existe_en_catalogo` = 2.089 (1.522 con código `9999`), `fecha_matricula_invalida` = 1, `campo_obligatorio_vacio` = 1, duplicados = 0. El CIIU vacío también cuenta como "no existe en el catálogo"; en la fuente solo ocurre en empresas canceladas.

In [ ]:
%sql
-- Evidencia: filas de Bronze que incumplirían cada regla (mismas condiciones que el Paso 2)
WITH catalogo AS (
  SELECT DISTINCT TRIM(code) AS code
  FROM datos_empresas.bronze.de_semiestructurado_actividadeseconomicas_api
  WHERE TRIM(code) RLIKE '^[0-9]{4}$' AND TRIM(COALESCE(description, '')) <> ''
),
r AS (
  SELECT b.*,
         TO_DATE(TRY_TO_TIMESTAMP(b.fecha_matricula, 'yyyyMMdd'))   AS fm,
         TO_DATE(TRY_TO_TIMESTAMP(b.fecha_cancelacion, 'yyyyMMdd')) AS fc,
         NULLIF(REGEXP_REPLACE(REGEXP_REPLACE(b.numero_identificacion, '[^0-9]', ''), '^0+', ''), '') AS id_normalizado,
         ROW_NUMBER() OVER (PARTITION BY TRIM(b.codigo_camara), TRIM(b.matricula)
                            ORDER BY TRY_TO_TIMESTAMP(b.fecha_actualizacion, 'yyyy/MM/dd HH:mm:ss.SSSSSSSSS') DESC NULLS LAST) AS version,
         c.code AS ciiu_en_catalogo
  FROM datos_empresas.bronze.de_semiestructurado_registromercantil_api b
  LEFT JOIN catalogo c ON c.code = TRIM(b.cod_ciiu_act_econ_pri)
)
SELECT
  COUNT(*) AS filas_bronze,
  SUM(CASE WHEN TRIM(COALESCE(codigo_camara, '')) = '' OR TRIM(COALESCE(matricula, '')) = ''
            OR TRIM(COALESCE(razon_social, '')) = '' OR TRIM(COALESCE(estado_matricula, '')) = '' THEN 1 ELSE 0 END) AS r1_campo_obligatorio_vacio,
  SUM(CASE WHEN id_normalizado IS NULL THEN 1 ELSE 0 END) AS r2_identificacion_invalida,
  SUM(CASE WHEN fm IS NULL OR fm < DATE'1900-01-01' OR fm > CURRENT_DATE() THEN 1 ELSE 0 END) AS r3_fecha_matricula_invalida,
  SUM(CASE WHEN fc < fm THEN 1 ELSE 0 END) AS r4_cancelacion_anterior_a_matricula,
  SUM(CASE WHEN ciiu_en_catalogo IS NULL THEN 1 ELSE 0 END) AS r5_ciiu_no_existe_en_catalogo,
  SUM(CASE WHEN version > 1 THEN 1 ELSE 0 END) AS r6_duplicado_version_anterior
FROM r;

---
## Paso 2: Reglas de calidad y separación Silver / DQL

Las reglas se evalúan **durante** el procesamiento: un registro rechazado nunca llega a Silver y, por lo tanto, tampoco a Gold.

In [ ]:
%python
llave_vacia = es_vacio("codigo_camara") | es_vacio("matricula")

REGLAS = [
    ("campo_obligatorio_vacio",
     llave_vacia | es_vacio("razon_social") | es_vacio("estado_matricula")),
    ("identificacion_invalida",
     F.col("numero_identificacion_normalizado").isNull()),
    ("fecha_matricula_invalida",
     F.col("fecha_matricula").isNull()
     | (F.col("fecha_matricula") < F.lit("1900-01-01").cast("date"))
     | (F.col("fecha_matricula") > F.current_date())),
    ("fecha_cancelacion_anterior_a_matricula",
     F.coalesce(F.col("fecha_cancelacion") < F.col("fecha_matricula"), F.lit(False))),
    ("ciiu_no_existe_en_catalogo",
     F.col("_ciiu_en_catalogo").isNull()),
    ("duplicado_version_anterior",
     (F.col("_version") > 1) & ~llave_vacia),
]

df_aceptados, df_rechazados = separar_aceptados_rechazados(df_versionado, REGLAS)

COLUMNAS_SILVER = [
    "codigo_camara", "camara_comercio", "matricula", "razon_social", "sigla",
    "clase_identificacion", "numero_identificacion", "numero_identificacion_normalizado",
    "nit", "digito_verificacion", "nit_normalizado",
    "codigo_ciiu_principal", "codigo_ciiu_secundario",
    "fecha_matricula", "fecha_renovacion", "ultimo_ano_renovado", "fecha_vigencia", "fecha_cancelacion",
    "tipo_sociedad", "organizacion_juridica", "categoria_matricula", "estado_matricula",
    "inscripcion_proponente", "fecha_actualizacion",
]

df_silver = construir_silver(df_aceptados, COLUMNAS_SILVER)
df_dql = construir_dql(df_rechazados)

---
## Paso 3: Escritura idempotente (MERGE) en Silver y DQL

In [ ]:
%python
m_silver = escribir_merge(
    df_silver, SILVER_RUES, ["codigo_camara", "matricula"], capa="silver",
    comentario="Silver - Registro mercantil RUES validado: version vigente de cada matricula por camara de comercio.",
)
m_dql = escribir_merge(
    df_dql, DQL_RUES, ["_id_registro_bronze"], capa="dql", columna_fecha="_fecha_rechazo",
    comentario="DQL - Registros de RUES rechazados, con la fila original de Bronze y el motivo.",
)

conteos = conciliar_fuente("RUES")

---
## Paso 4: Validación

In [ ]:
%sql
-- Rechazos por motivo (un registro puede incumplir varias reglas a la vez)
SELECT _motivo_rechazo, COUNT(*) AS registros
FROM datos_empresas.dql.de_registromercantil_dql
GROUP BY _motivo_rechazo
ORDER BY registros DESC;

In [ ]:
%sql
-- Códigos CIIU que le faltan al catálogo: insumo para corregir la fuente
SELECT cod_ciiu_act_econ_pri AS codigo_ciiu_no_encontrado, COUNT(*) AS empresas
FROM datos_empresas.dql.de_registromercantil_dql
WHERE _motivo_rechazo LIKE '%ciiu_no_existe_en_catalogo%'
GROUP BY cod_ciiu_act_econ_pri
ORDER BY empresas DESC
LIMIT 15;

In [ ]:
%sql
SELECT * FROM datos_empresas.silver.de_registromercantil LIMIT 10;

In [ ]:
%python
finalizar({"notebook": "03_Silver_RUES", "conteos": conteos, "silver": m_silver, "dql": m_dql})